### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="maternal_health_risk",
    dataset_year="2020",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5DP5D",
    download_description="""
We download the data from the UCI repository and unzip it to a predefined folder.

mkdir -p local-data-warehouse/maternal_health_risk/ && wget -P local-data-warehouse/maternal_health_risk/ https://archive.ics.uci.edu/static/public/863/maternal+health+risk.zip && unzip local-data-warehouse/maternal_health_risk/maternal+health+risk.zip -d local-data-warehouse/maternal_health_risk/
""",
    # References
    academic_reference_bibtex="""@inproceedings{ahmed2020review,
  title={Review and analysis of risk factor of maternal health in remote area using the Internet of Things (IoT)},
  author={Ahmed, Marzia and Kashem, Mohammod Abul and Rahman, Mostafijur and Khatun, Sabira},
  booktitle={InECCE2019: Proceedings of the 5th International Conference on Electrical, Control \& Computer Engineering, Kuantan, Pahang, Malaysia, 29th July 2019},
  pages={357--365},
  year={2020},
  organization={Springer}
}
""",
    academic_reference_bibtex_key="ahmed2020review",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- Anomaly: the data has a lot of duplicates (55%).
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="RiskLevel",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="RiskLevel",
)

## Preprocessing

In [ ]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/Maternal Health Risk Data Set.csv")

cat_features = [
    "RiskLevel",
]
df[cat_features] = df[cat_features].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()